## Aeropulse — Silver: Flight Fact

**Purpose:** Bronze → Silver for the flights fact table. Renames and casts columns, parses `flight_date` (US `M/d/yyyy h:mm:ss a` format), derives `is_cancelled` / `is_diverted` booleans and `flight_date_id`, applies cancellation-consistency DQ flags, deduplicates on the flight grain, and merges into `silver.flight`.

**Load type:** Incremental — filtered to the current `batch_id`, so re-running a batch only replaces that batch's own rows.

**Grain / key columns:** `flight_date`, `operating_carrier_code`, `flight_number`, `origin_airport_code` (validated as unique in the exploration notebook)

**Batch parameters:** `batch_id`, `batch_year` — set at the top of the notebook, one run per month of source data

**Depends on:** `silver-environment`, `silver-helper` (run via `%run`)

**Reads:** `aeropulse_bronze_lh.dbo.bronze_flight` (via `flight_bronze_path`)

**Writes:** `silver.flight` (merge on the grain above)

**Default lakehouse:** `aeropulse_silver_lh`


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 3, Finished, Available, Finished, False)

In [2]:
batch_id = ""
batch_year = ""

StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 4, Finished, Available, Finished, False)

In [3]:
%run silver-environment

StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 5, Finished, Available, Finished, True)

In [4]:
%run silver-helper

StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 8, Finished, Available, Finished, True)

In [5]:
flight_df = spark.read.format('delta').load(flight_bronze_path).filter(F.col("batch_id") == batch_id)

StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 9, Finished, Available, Finished, False)

In [6]:
flight_rename_map = {
    'FL_DATE': 'flight_date',
    'OP_UNIQUE_CARRIER': 'operating_carrier_code',
    'TAIL_NUM': 'tail_number',
    'OP_CARRIER_FL_NUM': 'flight_number',
    'ORIGIN': 'origin_airport_code',
    'ORIGIN_CITY_NAME': 'origin_city_name',
    'DEST': 'destination_airport_code',
    'DEST_CITY_NAME': 'destination_city_name',
    'DEP_TIME': 'departure_time',
    'DEP_DELAY': 'departure_delay_minutes',
    'TAXI_OUT': 'taxi_out_minutes',
    'WHEELS_OFF': 'wheels_off_time',
    'WHEELS_ON': 'wheels_on_time',
    'TAXI_IN': 'taxi_in_minutes',
    'ARR_TIME': 'arrival_time',
    'ARR_DELAY': 'arrival_delay_minutes',
    'CANCELLED': 'cancelled',
    'CANCELLATION_CODE': 'cancellation_code',
    'DIVERTED': 'diverted',
    'AIR_TIME': 'air_time_minutes',
    'DISTANCE': 'distance_miles',
    'CARRIER_DELAY': 'carrier_delay_minutes',
    'WEATHER_DELAY': 'weather_delay_minutes',
    'NAS_DELAY': 'nas_delay_minutes',
    'SECURITY_DELAY': 'security_delay_minutes',
    'LATE_AIRCRAFT_DELAY': 'late_aircraft_delay_minutes',
    'batch_id': 'batch_id',
    'ingested_timestamp': 'ingested_timestamp',
    'source_path': 'source_path',
}

flight_cast_map = {
    "departure_delay_minutes": "double", "arrival_delay_minutes": "double",
    "taxi_out_minutes": "double", "taxi_in_minutes": "double",
    "air_time_minutes": "double", "distance_miles": "double",
    "carrier_delay_minutes": "double", "weather_delay_minutes": "double",
    "nas_delay_minutes": "double", "security_delay_minutes": "double",
    "late_aircraft_delay_minutes": "double",
    "departure_time": "int", "arrival_time": "int",
    "wheels_off_time": "int", "wheels_on_time": "int"
}

key_columns = ["flight_date", "operating_carrier_code", "flight_number", "origin_airport_code"]

flight_df = (flight_df
    .transform(trim_whitespaces)
    .transform(lambda d: rename_column(d, flight_rename_map))
)


flight_df = remove_nulls(flight_df, key_columns + ["destination_airport_code"])
flight_df = remove_duplicates(flight_df, key_columns)
flight_df = cast_columns(flight_df, flight_cast_map)

##display(flight_df.limit(10))





StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 10, Finished, Available, Finished, False)

In [7]:
flight_df = (flight_df
    .transform(lambda d: cast_columns(d, flight_cast_map))
    .withColumn("flight_date", F.to_date(F.col("flight_date"), "M/d/yyyy h:mm:ss a"))
    .withColumn("flight_date_id", F.regexp_replace(F.col("flight_date"), "-", ""))
    .withColumn("is_cancelled", F.col("cancelled").cast("double") == 1)
    .withColumn("is_diverted", F.col("diverted").cast("double") == 1)
    .drop("cancelled", "diverted")
)


StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9c3cd2e-6954-4f44-8c12-db290029245f)

In [8]:

flight_df = add_dq_flag(
    flight_df,
    F.col("is_cancelled") & (F.col("departure_time").isNotNull() | F.col("arrival_time").isNotNull()),
    "dq_cancelled_with_time_flag"
)
flight_df = add_dq_flag(
    flight_df,
    (~F.col("is_cancelled")) & F.col("cancellation_code").isNotNull(),
    "dq_non_cancelled_with_code_flag"
)

flight_df = add_sk_key(flight_df, key_columns, "flight_sk")


StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1a44bfb-2434-4006-88ca-12ee44161c43)

In [9]:
merge_condition = " AND ".join([f"s.{c} = t.{c}" for c in key_columns])
update_cols = [c for c in flight_df.columns if c not in key_columns]

write_to_silver(
    flight_df,
    "silver.flight",
    merge_condition,
    update_cols
)

StatementMeta(, e28bb0ee-cdca-4ba6-ad38-26d4fb90d1cb, 13, Finished, Available, Finished, False)